In [ ]:
#Install dependencies (no bitsandbytes needed)
!pip install --quiet transformers torch firebase-admin
from transformers import pipeline
!pip install bitsandbytes --upgrade
!pip install --upgrade bitsandbytes


  Using cached bitsandbytes-0.48.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.48.2-py3-none-manylinux_2_24_x86_64.whl (59.4 MB)


In [ ]:
!pip install -U bitsandbytes


In [ ]:
#Import libraries
import firebase_admin
from firebase_admin import credentials, firestore
from transformers import pipeline
import torch
import time
import huggingface_hub
import transformers
print(f"Hugging Face Hub Version: {huggingface_hub.__version__}")
print(f"Transformers Version: {transformers.__version__}")

Hugging Face Hub Version: 0.36.0
Transformers Version: 4.57.1


In [ ]:
#Initialize Firebase
import os
from google.colab import files
import torch


if not firebase_admin._apps:  # Prevent "already initialized" error
    service_account_path = "/content/serviceAccountKey.json"
    if not os.path.exists(service_account_path):
        print("Service account key not found. Please upload 'serviceAccount.json'.")
        uploaded = files.upload()
        if 'serviceAccount.json' not in uploaded:
            print("Error: 'serviceAccount.json' was not uploaded. Please ensure the file is named correctly and try again.")
            raise FileNotFoundError(f"Expected 'serviceAccount.json' but found {list(uploaded.keys())}")

    cred = credentials.Certificate(service_account_path)
    firebase_admin.initialize_app(cred)

# Firestore client
db = firestore.client()

In [ ]:
import torch
from transformers import pipeline, BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer # Added AutoModelForCausalLM, AutoTokenizer

# ------------------- Sentiment Analysis Model -------------------
# Model for analyzing journals
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)

# ------------------- Feedback Model -------------------
# Model for generating short insights or encouragement
feedback_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    device=0 if torch.cuda.is_available() else -1
)

# ------------------- Chat Model -------------------
chat_model_name = "tiiuae/falcon-7b-instruct"

def load_chat_model():
    """Loads the chat model safely with GPU support if available."""
    print(f"Loading chat model: {chat_model_name}...")

    # Configure 4-bit quantization for efficient loading of large models
    # This is often necessary for large models like Falcon-7B on Colab GPUs.
    nf4_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 # Use bfloat16 for better performance on modern GPUs
    )

    try:
        # Load model and tokenizer separately to ensure quantization_config is only used during loading
        model = AutoModelForCausalLM.from_pretrained(
            chat_model_name,
            quantization_config=nf4_config,
            device_map="auto",
            trust_remote_code=True
        )
        tokenizer = AutoTokenizer.from_pretrained(chat_model_name, trust_remote_code=True)

        chat_pipeline = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer
        )
        print("✅ Chat model loaded successfully with 4-bit quantization on GPU.")
    except Exception as e:
        print(f"⚠️ Error loading chat model with 4-bit quantization on GPU: {e}")
        print("Trying to load on CPU instead (might be very slow)...")
        # If GPU loading fails even with quantization, try CPU (which might still run out of RAM if not handled carefully by accelerate)
        chat_pipeline = pipeline(
            "text-generation",
            model=chat_model_name,
            trust_remote_code=True,
            device=-1  # Force CPU
        )
        print("✅ Chat model loaded successfully on CPU (without quantization).")

    return chat_pipeline

# Load the chat model
chat_pipeline = load_chat_model()

print("✅ All models loaded successfully.")


Device set to use cpu
Device set to use cpu


Loading chat model: tiiuae/falcon-7b-instruct...
⚠️ Error loading chat model with 4-bit quantization on GPU: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`
Trying to load on CPU instead (might be very slow)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cpu


✅ Chat model loaded successfully on CPU (without quantization).
✅ All models loaded successfully.


In [ ]:
#Function to analyze a single journal
def analyze_journal(doc, db):
    data = doc.to_dict()
    entry_text = data.get('entry', '')
    if not entry_text:
        return

    # Skip if sentiment already exists
    if data.get('sentiment') is not None:
        return

    # Run sentiment analysis
    result = sentiment_model(entry_text)[0]
    label = result['label']
    score = result['score']
    normalized = score if label == 'POSITIVE' else -score

    # Update Firestore
    db.collection('journals').document(doc.id).update({
        'sentiment': {
            'label': label,
            'score': score,
            'normalized': normalized
        }
    })
    print(f"Updated {doc.id}: {label}, {score}")

In [ ]:
#Process all existing journals
def analyze_existing_journals(db):
    journals_ref = db.collection('journals')
    for doc in journals_ref.stream():
        analyze_journal(doc, db)
        time.sleep(0.1)  # small delay to avoid throttling

In [ ]:
#Listen for new journals automatically
def listen_for_new_journals(db):
    journals_ref = db.collection('journals').where('sentiment', '==', None)

    def on_snapshot(docs, changes, read_time):
        for change in changes:
            if change.type.name in ('ADDED', 'MODIFIED'):
                analyze_journal(change.document, db)

    journals_ref.on_snapshot(on_snapshot)
    print("Listening for new journals...")

In [ ]:

# Function to answer anonymous questions
def answer_questions(db):
    print("Processing open questions...")
    qna_ref = db.collection('qna').where('status', '==', 'open')
    docs = qna_ref.stream()

    for doc in docs:
        data = doc.to_dict()
        question_text = data.get('question', '')
        if question_text:
            try:
                # Generate answer (truncate question to 128 tokens for speed)
                response = chat_pipeline(
                    question_text[:128],
                    max_length=200,
                    do_sample=True,
                    truncation=True # Added truncation=True to address warning
                )[0]['generated_text']

                # Update Firestore with response
                db.collection('qna').document(doc.id).update({
                    'response': response,
                    'status': 'answered',
                    'answered_at': firestore.SERVER_TIMESTAMP
                })
                print(f"✅ Answered question {doc.id}")
            except Exception as e:
                print(f"❌ Error processing question {doc.id}: {e}")
                db.collection('qna').document(doc.id).update({'response': f"Error: {e}"})
    print("✅ Question answering complete.")


In [ ]:
# 1️⃣ Process existing journals
analyze_existing_journals(db)

# 2️⃣ Start listening for new journals
listen_for_new_journals(db)

Listening for new journals...


/usr/local/lib/python3.12/dist-packages/google/cloud/firestore_v1/base_collection.py:304: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)


In [ ]:
# Start listening for new questions
answer_questions(db)

Processing open questions...


/usr/local/lib/python3.12/dist-packages/google/cloud/firestore_v1/base_collection.py:304: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❌ Error processing question WBjgFd6TUNPHse9QyLon: 'NoneType' object has no attribute 'shape'
✅ Question answering complete.


In [ ]:
from google.cloud import firestore
from transformers import pipeline
import datetime
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/serviceAccountKey.json"


# ------------------- Setup -------------------
# Initialize Firestore
db = firestore.Client()  # Make sure GOOGLE_APPLICATION_CREDENTIALS is set

# Initialize the lightweight feedback model
feedback_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
)

# ------------------- Function -------------------
def analyze_all_journals():
    """
    Analyze all existing journal entries in Firestore and add AI feedback.
    """
    journals_ref = db.collection('journals')
    docs = journals_ref.stream()

    for doc in docs:
        data = doc.to_dict()
        entry_text = data.get('entry', '')

        # Skip if empty or already analyzed
        if not entry_text or data.get('ai_feedback') is not None:
            continue

        # Generate encouragement / insight
        prompt = f"Read the following journal entry and give a short, empathetic insight or encouragement:\n\n{entry_text}"
        result = feedback_pipeline(prompt, max_new_tokens=60)
        ai_feedback = result[0]['generated_text']

        # Timestamp
        analyzed_at = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Update Firestore
        journals_ref.document(doc.id).update({
            'ai_feedback': ai_feedback,
            'analyzed_at': analyzed_at
        })

        print(f"✅ Updated {doc.id} with feedback: {ai_feedback}")

# ------------------- Run -------------------
analyze_all_journals()


Device set to use cpu


✅ Updated 4PPeUMCB0VXNM4kQ2gZm with feedback: a few days.
